## Experiment 4: Representation Analysis

Testing the core research question: do World A and World B's SAEs learn different representations, given matched observable statistics? Checked via four progressively more rigorous metrics: raw feature-recovery correlation, partial correlation (controlling for the A-B relationship)best-latent-per-seed comparison, and representation geometry.

In [1]:
import sys
sys.path.append('..')

import numpy as np
import torch
from src.generators import generate_world_a, generate_world_b
from src.sae import SparseAutoencoder
from src.training import train_sae, train_multiple_seeds
from src.metrics import compute_feature_recovery

In [2]:
results_a, X_tensor_a = train_multiple_seeds(generate_world_a, {}, n_seeds=5)

Epoch 0: total=0.3799  recon=0.3113  sparsity=2.2889
Epoch 20: total=0.3171  recon=0.2568  sparsity=2.0109
Epoch 40: total=0.2761  recon=0.2199  sparsity=1.8712
Epoch 60: total=0.2461  recon=0.1912  sparsity=1.8311
Epoch 80: total=0.2220  recon=0.1664  sparsity=1.8527
Epoch 100: total=0.2026  recon=0.1456  sparsity=1.8997
Epoch 120: total=0.1863  recon=0.1281  sparsity=1.9377
Epoch 140: total=0.1722  recon=0.1125  sparsity=1.9880
Epoch 160: total=0.1599  recon=0.0991  sparsity=2.0265
Epoch 180: total=0.1493  recon=0.0883  sparsity=2.0361
Seed 0: final_recon=0.0802  active_latents=6.60
Epoch 0: total=0.4016  recon=0.2870  sparsity=3.8214
Epoch 20: total=0.3389  recon=0.2320  sparsity=3.5634
Epoch 40: total=0.2966  recon=0.1950  sparsity=3.3835
Epoch 60: total=0.2649  recon=0.1675  sparsity=3.2450
Epoch 80: total=0.2385  recon=0.1445  sparsity=3.1332
Epoch 100: total=0.2161  recon=0.1251  sparsity=3.0332
Epoch 120: total=0.1961  recon=0.1073  sparsity=2.9613
Epoch 140: total=0.1787  reco

### Metric 1: Raw Feature Recovery

Correlating each latent's activation with ground-truth A and B. Some latents are "dead" (never activate, std=0) and are left as NaN rather than causing a division error.

In [3]:
z0 = results_a[0]["z"]
X_a = generate_world_a(n_samples=1000, seed=42)
X_a_tensor = torch.tensor(X_a, dtype=torch.float32)

corr_a, corr_b = compute_feature_recovery(z0, X_a_tensor)
print("Correlation with A:", np.round(corr_a, 2))
print("Correlation with B:", np.round(corr_b, 2))

Correlation with A: [ 0.64  0.73 -0.11 -0.54  0.36   nan  0.01 -0.11 -0.3    nan  0.33 -0.13
  0.82  0.19 -0.16  0.44  0.63   nan  0.42 -0.29]
Correlation with B: [ 0.69  0.84 -0.12 -0.56  0.39   nan  0.22 -0.12 -0.26   nan  0.31 -0.1
  0.88  0.12 -0.24  0.45  0.59   nan  0.44 -0.59]


In [4]:
results_b, X_tensor_b = train_multiple_seeds(generate_world_b, {}, n_seeds=5)

z0_b = results_b[0]["z"]
X_b = generate_world_b(n_samples=1000, seed=42)
X_b_tensor = torch.tensor(X_b, dtype=torch.float32)

corr_a_b, corr_b_b = compute_feature_recovery(z0_b, X_b_tensor)
print("World B — Correlation with A:", np.round(corr_a_b, 2))
print("World B — Correlation with B:", np.round(corr_b_b, 2))

Epoch 0: total=0.3899  recon=0.3201  sparsity=2.3256
Epoch 20: total=0.3251  recon=0.2639  sparsity=2.0421
Epoch 40: total=0.2827  recon=0.2258  sparsity=1.8962
Epoch 60: total=0.2516  recon=0.1960  sparsity=1.8539
Epoch 80: total=0.2269  recon=0.1703  sparsity=1.8897
Epoch 100: total=0.2077  recon=0.1492  sparsity=1.9482
Epoch 120: total=0.1920  recon=0.1322  sparsity=1.9931
Epoch 140: total=0.1781  recon=0.1172  sparsity=2.0297
Epoch 160: total=0.1656  recon=0.1040  sparsity=2.0536
Epoch 180: total=0.1546  recon=0.0931  sparsity=2.0525
Seed 0: final_recon=0.0845  active_latents=6.88
Epoch 0: total=0.4045  recon=0.2886  sparsity=3.8651
Epoch 20: total=0.3412  recon=0.2332  sparsity=3.6006
Epoch 40: total=0.2989  recon=0.1966  sparsity=3.4083
Epoch 60: total=0.2673  recon=0.1697  sparsity=3.2552
Epoch 80: total=0.2411  recon=0.1470  sparsity=3.1357
Epoch 100: total=0.2187  recon=0.1277  sparsity=3.0355
Epoch 120: total=0.1990  recon=0.1101  sparsity=2.9633
Epoch 140: total=0.1818  reco

## Note: An Invalid Averaging Approach

An earlier version of this analysis averaged partial correlations across seeds by raw latent index (e.g., "latent 7's average correlation across all 5 seeds").This is methodologically invalid — each seed's SAE has independently random initialization, so latent index 7 in seed 0 has no reason to represent the same thing as latent index 7 in seed 3. Averaging across mismatched latents produces 
a number that doesn't correspond to anything real in either world.

Fixed by comparing each seed's *best-matching* latent instead (max partial correlation per seed, below) - this sidesteps the index-alignment problem entirely and gives a methodologically sound comparison.

### Metric 2: Partial Correlation (Corrected)

Controlling for each variable's correlation with the other, then comparing 
each seed's best-matching latent across worlds — the methodologically sound 
version, after fixing the averaging issue above.

In [5]:
from src.metrics import max_partial_correlation_per_seed

max_a_A, max_b_A = max_partial_correlation_per_seed(results_a, X_a_tensor)
max_a_B, max_b_B = max_partial_correlation_per_seed(results_b, X_b_tensor)

print("World A — max partial corr with A per seed:", np.round(max_a_A, 3))
print("World B — max partial corr with A per seed:", np.round(max_a_B, 3))
print("\nWorld A — max partial corr with B per seed:", np.round(max_b_A, 3))
print("World B — max partial corr with B per seed:", np.round(max_b_B, 3))

World A — max partial corr with A per seed: [0.444 0.78  0.464 0.604 0.688]
World B — max partial corr with A per seed: [0.471 0.721 0.524 0.553 0.649]

World A — max partial corr with B per seed: [0.666 0.496 0.555 0.801 0.643]
World B — max partial corr with B per seed: [0.699 0.508 0.564 0.8   0.628]


### Metric 3: Representation Geometry

Comparing the overall structure of inter-latent correlations (how entangled/clustered latents are with each other), summarized as mean absolute off-diagonal correlation per seed.

In [6]:
from src.metrics import summarize_geometry, latent_correlation_matrix

geo_a = [summarize_geometry(latent_correlation_matrix(r["z"])) for r in results_a]
geo_b = [summarize_geometry(latent_correlation_matrix(r["z"])) for r in results_b]

print("World A — mean |off-diagonal correlation| per seed:", np.round(geo_a, 3))
print("World B — mean |off-diagonal correlation| per seed:", np.round(geo_b, 3))

World A — mean |off-diagonal correlation| per seed: [0.251 0.279 0.24  0.28  0.291]
World B — mean |off-diagonal correlation| per seed: [0.246 0.275 0.234 0.283 0.291]


## Result

Across all four metrics - raw correlation, partial correlation, best-latent-per-seed, and geometry;World A and World B produce statistically indistinguishable SAE representations, differing only within normal seed-to-seed noise (~0.01–0.06). This supports **H0**: under these controlled 
conditions (matched observable statistics, small linear SAE, binary synthetic features), the underlying causal generative structure does not leave a detectable signature in the learned representation.

See Limitations in the final report for what this result does and does not establish.